In [0]:
catalog_name = "customer_360"
volume_name = "raw"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}"),
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}"),
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}"),
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}"),
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{bronze_schema}.{volume_name}")

In [0]:
landing_folder_path = "/Volumes/customer_360/bronze/raw/customer_360_project_data/"

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

def get_table_name(table_name):
  return f"{catalog_name}.{bronze_schema}.{table_name}"

def get_volume_path(volume_name):
  return f"{catalog_name}.{volume_name}"

def add_ingestion_metadata(df, source_file):
  return df.withColumn("_ingestion_timestamp", f.current_timestamp()) \
           .withColumn("_source_file", f.lit(source_file))

def load_to_bronze(table_name, source_path, df):
  watermark_table = f"{catalog_name}.{gold_schema}.pipeline_watermark"
  
  watermark_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("processed_timestamp", TimestampType(), False)
  ])
  spark.createDataFrame([], watermark_schema).write.format("delta").mode("ignore").saveAsTable(watermark_table)

  # Extract file name from source path
  file_name = source_path.split("/")[-1]

  # Check if this file has already been processed
  already_processed = (
    spark.table(watermark_table)
    .filter(
      (f.col("table_name") == table_name) &
      (f.col("file_name") == file_name)
    )
    .limit(1)
    .count()
  )

  if already_processed > 0:
    print(f"Skipping {file_name} - already loaded into {table_name}")
    return False

  try:
    # Append to bronze table
    df.write.mode("append").option("mergeSchema", "true").saveAsTable(table_name)

    # Record in watermark table
    watermark_entry = (
      spark.createDataFrame(
        [(table_name, file_name)],
        ["table_name", "file_name"]
      ).withColumn("processed_timestamp", f.current_timestamp())
    )
    watermark_entry.write.format("delta").mode("append").saveAsTable(watermark_table)

    print(f"Loaded {file_name} -> {table_name}")
    return True

  except Exception as e:
    print(f"Error loading {file_name} -> {table_name}: {e}")
    # Rollback: delete rows that were just appended
    try:
      spark.sql(f"DELETE FROM {table_name} WHERE _source_file = '{source_path}'")
      print(f"Rollback: removed rows from {table_name} for {file_name}")
    except Exception as rollback_e:
      print(f"Rollback failed for {file_name}: {rollback_e}")
    return False